In [36]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time

# --- Настройка страницы ---
st.set_page_config(page_title="Сравнение моделей F1", layout="wide")
st.title("🏎️ Сравнение моделей машинного обучения на данных о победах в Формуле-1")
st.markdown("Выберите одну или несколько моделей, настройте гиперпараметры, обучите и сравните результаты.")

# --- Загрузка и предобработка данных (с учётом вашего CSV) ---
@st.cache_data
def load_and_preprocess():
    try:
        df = pd.read_csv("race_wins_1950-2020.csv", index_col=0)
    except FileNotFoundError:
        st.error("Файл 'race_wins_1950-2020.csv' не найден. Загрузите его в директорию Colab.")
        st.stop()

    # Преобразование года
    df['Year'] = pd.to_datetime(df['Date'], format='%d.%m.%Y').dt.year

    # Функция преобразования времени в секунды
    def time_to_seconds(t):
        if pd.isna(t) or not isinstance(t, str):
            return np.nan
        t = t.strip()
        parts = t.split(':')
        try:
            if len(parts) == 3:
                h, m, s = parts
                return int(h)*3600 + int(m)*60 + float(s)
            elif len(parts) == 2:
                m, s = parts
                return int(m)*60 + float(s)
            else:
                return np.nan
        except:
            return np.nan

    df['TimeSeconds'] = df['Time'].apply(time_to_seconds)
    df_clean = df.dropna(subset=['TimeSeconds', 'Laps']).copy()
    st.success(f"✅ Данные загружены. После очистки: {len(df_clean)} записей (удалено {len(df)-len(df_clean)} строк с пропусками)")
    return df_clean

df = load_and_preprocess()

# --- Боковая панель: выбор признаков, моделей, гиперпараметров ---
st.sidebar.header("⚙️ Параметры эксперимента")

# Признаки
available_features = ['Year', 'Laps']   # можно расширить, добавив Venue с one-hot, но для простоты оставляем числовые
feature_options = st.sidebar.multiselect(
    "Признаки для модели",
    options=available_features,
    default=available_features,
    help="Выберите колонки для предсказания времени гонки"
)
if not feature_options:
    st.sidebar.error("Выберите хотя бы один признак!")
    st.stop()

target = 'TimeSeconds'
st.sidebar.markdown(f"**Целевая переменная:** `{target}` (время в секундах)")

# Выбор моделей
available_models = {
    "Linear Regression": LinearRegression,
    "Random Forest": RandomForestRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    "SVR": SVR,
    "KNN": KNeighborsRegressor
}
selected_models = st.sidebar.multiselect(
    "Модели (можно несколько)",
    options=list(available_models.keys()),
    default=["Linear Regression", "Random Forest"]
)
if not selected_models:
    st.sidebar.error("Выберите хотя бы одну модель!")
    st.stop()

# Гиперпараметры для выбранных моделей
st.sidebar.subheader("🎛️ Гиперпараметры")
test_size = st.sidebar.slider("Доля тестовой выборки", 0.1, 0.4, 0.2, step=0.05)
random_state = 42

params = {}
for model_name in selected_models:
    with st.sidebar.expander(f"Параметры: {model_name}"):
        if model_name == "Linear Regression":
            params[model_name] = {}
        elif model_name == "Random Forest":
            n_estimators = st.slider("n_estimators", 10, 200, 100, 10, key=f"rf_n_{model_name}")
            max_depth = st.slider("max_depth", 1, 20, 10, key=f"rf_d_{model_name}")
            params[model_name] = {"n_estimators": n_estimators, "max_depth": max_depth, "random_state": random_state}
        elif model_name == "Gradient Boosting":
            n_estimators = st.slider("n_estimators", 10, 200, 100, 10, key=f"gb_n_{model_name}")
            learning_rate = st.number_input("learning_rate", 0.01, 1.0, 0.1, 0.01, key=f"gb_lr_{model_name}")
            max_depth = st.slider("max_depth", 1, 10, 3, key=f"gb_d_{model_name}")
            params[model_name] = {"n_estimators": n_estimators, "learning_rate": learning_rate,
                                  "max_depth": max_depth, "random_state": random_state}
        elif model_name == "SVR":
            C = st.number_input("C", 0.1, 100.0, 1.0, 0.1, key=f"svr_c_{model_name}")
            epsilon = st.number_input("epsilon", 0.01, 1.0, 0.1, 0.01, key=f"svr_e_{model_name}")
            params[model_name] = {"C": C, "epsilon": epsilon, "kernel": "rbf"}
        elif model_name == "KNN":
            n_neighbors = st.slider("n_neighbors", 1, 50, 5, 1, key=f"knn_n_{model_name}")
            params[model_name] = {"n_neighbors": n_neighbors}

train_button = st.sidebar.button("🚀 Обучить выбранные модели", type="primary")

# --- Основная область: обучение и результаты ---
if train_button:
    # Подготовка данных
    X = df[feature_options].copy()
    y = df[target]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=random_state
    )

    st.subheader("📈 Результаты обучения")
    results = []
    predictions = {}
    models_objects = {}

    for model_name in selected_models:
        with st.spinner(f"Обучение {model_name}..."):
            start = time.time()
            model = available_models[model_name](**params[model_name])
            model.fit(X_train, y_train)
            train_time = time.time() - start

            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            predictions[model_name] = y_pred_test
            models_objects[model_name] = model

            results.append({
                "Модель": model_name,
                "Время обучения (сек)": round(train_time, 3),
                "Train MAE": round(mean_absolute_error(y_train, y_pred_train), 2),
                "Train RMSE": round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 2),
                "Train R²": round(r2_score(y_train, y_pred_train), 3),
                "Test MAE": round(mean_absolute_error(y_test, y_pred_test), 2),
                "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 2),
                "Test R²": round(r2_score(y_test, y_pred_test), 3)
            })

    results_df = pd.DataFrame(results)
    st.dataframe(results_df, use_container_width=True)

    # --- График сравнения метрик (MAE и R²) ---
    fig_metrics = go.Figure()
    fig_metrics.add_trace(go.Bar(x=results_df["Модель"], y=results_df["Test MAE"],
                                 name="Test MAE (сек)", marker_color='coral', yaxis="y1"))
    fig_metrics.add_trace(go.Bar(x=results_df["Модель"], y=results_df["Test R²"],
                                 name="Test R²", marker_color='steelblue', yaxis="y2"))
    fig_metrics.update_layout(
        title="Сравнение метрик на тестовой выборке",
        xaxis_title="Модель",
        yaxis=dict(title="MAE (сек)", side="left"),
        yaxis2=dict(title="R²", overlaying="y", side="right", rangemode="tozero"),
        barmode='group',
        legend=dict(x=0.02, y=0.98)
    )
    st.plotly_chart(fig_metrics, use_container_width=True)

    # --- Детальный анализ предсказаний для выбранной модели ---
    st.subheader("🔍 Детальный анализ предсказаний")
    selected_model_for_plot = st.selectbox(
        "Выберите модель для просмотра графиков",
        options=selected_models
    )

    if selected_model_for_plot in predictions:
        y_pred = predictions[selected_model_for_plot]
        # 1. Scatter plot: предсказанные vs реальные
        fig_scatter = px.scatter(
            x=y_test, y=y_pred,
            labels={'x': 'Реальное время (сек)', 'y': 'Предсказанное время (сек)'},
            title=f"{selected_model_for_plot}: предсказанные vs реальные (тест)",
            trendline="ols",
            opacity=0.6
        )
        fig_scatter.add_shape(type="line", x0=y_test.min(), y0=y_test.min(),
                              x1=y_test.max(), y1=y_test.max(),
                              line=dict(color="red", dash="dash"))
        st.plotly_chart(fig_scatter, use_container_width=True)

        # 2. График остатков
        residuals = y_test - y_pred
        fig_resid = px.scatter(
            x=y_pred, y=residuals,
            labels={'x': 'Предсказанные значения (сек)', 'y': 'Остатки (сек)'},
            title=f"{selected_model_for_plot}: график остатков",
            opacity=0.6
        )
        fig_resid.add_hline(y=0, line_dash="dash", line_color="red")
        st.plotly_chart(fig_resid, use_container_width=True)

        # 3. Гистограмма остатков
        fig_hist = px.histogram(
            x=residuals, nbins=30,
            labels={'x': 'Остатки (сек)', 'y': 'Частота'},
            title=f"{selected_model_for_plot}: распределение остатков",
            opacity=0.7
        )
        st.plotly_chart(fig_hist, use_container_width=True)

        # 4. Важность признаков (для деревьев) или коэффициенты (для линейной регрессии)
        model_obj = models_objects[selected_model_for_plot]
        if hasattr(model_obj, 'feature_importances_'):
            importances = model_obj.feature_importances_
            feat_imp = pd.DataFrame({'Признак': feature_options, 'Важность': importances})
            feat_imp = feat_imp.sort_values('Важность', ascending=False)
            fig_imp = px.bar(feat_imp, x='Важность', y='Признак', orientation='h',
                             title=f"{selected_model_for_plot}: важность признаков", text_auto=True)
            st.plotly_chart(fig_imp, use_container_width=True)
        elif selected_model_for_plot == "Linear Regression" and hasattr(model_obj, 'coef_'):
            coefs = model_obj.coef_
            feat_imp = pd.DataFrame({'Признак': feature_options, 'Коэффициент': coefs})
            fig_coef = px.bar(feat_imp, x='Коэффициент', y='Признак', orientation='h',
                              title=f"{selected_model_for_plot}: коэффициенты", text_auto=True)
            st.plotly_chart(fig_coef, use_container_width=True)
        else:
            st.info("Для выбранной модели нет важности признаков или коэффициентов.")

    # --- Лучшая модель по R² ---
    best_model = results_df.loc[results_df["Test R²"].idxmax()]
    st.success(f"🏆 Лучшая модель по R² на тесте: **{best_model['Модель']}** (R² = {best_model['Test R²']:.3f})")

else:
    st.info("Настройте параметры в боковой панели и нажмите кнопку **Обучить выбранные модели**.")




Overwriting app.py


In [37]:
import subprocess
import time
import threading
from google.colab.output import eval_js

def run():
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.enableCORS", "false"])

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(5)

url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"\n✅ Откройте: {url}")


✅ Откройте: https://8501-m-s-kkb-use1c1-ihy4vs28g3sv-c.us-east1-1.prod.colab.dev
